
# Movimiento Circular


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Movimiento Circular Uniforme (MCU)</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">Ingresa el radio y la velocidad angular. Observa cómo el vector velocidad siempre es tangente y la aceleración centrípeta apunta al centro.</p>
    
    <!-- Controles de Parámetros -->
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center;">
        <div>
            <label style="font-weight: bold;">Radio (r, m): </label>
            <input type="number" id="radio_UID" value="2.0" step="0.5" min="0.5" style="width: 70px; padding: 5px; border-radius: 4px; border: 1px solid #ccc;">
        </div>
        <div>
            <label style="font-weight: bold;">Velocidad angular (ω, rad/s): </label>
            <input type="number" id="omega_UID" value="1.5" step="0.1" min="0.1" style="width: 70px; padding: 5px; border-radius: 4px; border: 1px solid #ccc;">
        </div>
        <button id="update_params_UID" style="background-color: #0d6efd; color: white; border: none; padding: 8px 15px; border-radius: 5px; cursor: pointer; font-weight: bold;">Actualizar Parámetros</button>
    </div>

    <!-- Controles de Reproducción -->
    <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 15px; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir / Pausar</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer;">⏹ Reiniciar</button>
            <div style="font-size: 1.1em; font-weight: bold; margin-left: 10px;">
                Tiempo: <span id="time_display_UID" style="color: #dc3545;">0.00</span> s
            </div>
        </div>
        <div style="display: flex; align-items: center; gap: 10px; background: #e9ecef; padding: 8px 15px; border-radius: 5px;">
            <label style="font-weight: 600; font-size: 0.9em; margin: 0;">Velocidad Animación:</label>
            <input type="range" id="speed_UID" min="0.1" max="1.5" step="0.1" value="0.5" style="vertical-align: middle;">
            <span id="speed_val_UID" style="font-weight: bold; font-family: monospace; width: 35px;">0.5x</span>
        </div>
    </div>
    
    <!-- Contenedor Principal (Simulación + Gráficos) -->
    <div style="display: flex; gap: 20px; flex-wrap: wrap;">
        <!-- Canvas de la persecución física -->
        <div style="flex: 1; min-width: 300px; max-width: 350px;">
            <canvas id="canvas_sim_UID" width="350" height="400" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); width: 100%;"></canvas>
        </div>
        <!-- Canvas de los gráficos -->
        <div style="flex: 2; min-width: 450px;">
            <canvas id="canvas_graphs_UID" width="600" height="400" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); width: 100%;"></canvas>
        </div>
    </div>
    
    <div id="results_UID" style="margin-top: 15px; font-size: 0.95em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41;">
        <!-- Resultados dinámicos -->
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvasSim = document.getElementById('canvas_sim_UID');
            const canvasGraph = document.getElementById('canvas_graphs_UID');
            if (!canvasSim || !canvasGraph) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctxSim = canvasSim.getContext('2d');
            const ctxG = canvasGraph.getContext('2d');
            
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_UID');
            const btn_update = document.getElementById('update_params_UID');
            const time_display = document.getElementById('time_display_UID');
            const speed_slider = document.getElementById('speed_UID');
            const speed_val = document.getElementById('speed_val_UID');
            const results_div = document.getElementById('results_UID');
            
            // Entradas
            const input_r = document.getElementById('radio_UID');
            const input_omega = document.getElementById('omega_UID');

            let isPlaying = false;
            let animationId;
            let lastTime = 0; 
            let t = 0;
            
            // Físicas
            let r, omega, v, ac, T, freq, tMax;

            function calculatePhysics() {
                r = parseFloat(input_r.value);
                omega = parseFloat(input_omega.value);
                
                if(r <= 0) r = 0.5;
                if(omega <= 0) omega = 0.1;
                input_r.value = r;
                input_omega.value = omega;

                v = omega * r;
                ac = omega * omega * r;
                T = 2 * Math.PI / omega;
                freq = 1 / T;
                
                // Animamos hasta 2 revoluciones completas para ver bien las gráficas
                tMax = 2 * T;

                results_div.innerHTML = `
                    <strong>Resultados calculados:</strong><br>
                    • Rapidez Tangencial (|v|): <b style="color:#0d6efd;">${v.toFixed(2)} m/s</b><br>
                    • Aceleración Centrípeta (|a_c|): <b style="color:#dc3545;">${ac.toFixed(2)} m/s²</b><br>
                    • Período (T): <b>${T.toFixed(2)} s</b> | Frecuencia (f): <b>${freq.toFixed(2)} Hz</b>
                `;
            }

            // Dibuja flechas con estilo en el canvas
            function drawArrow(ctx, fromx, fromy, tox, toy, color) {
                const headlen = 10;
                const angle = Math.atan2(toy - fromy, tox - fromx);
                
                ctx.beginPath();
                ctx.strokeStyle = color;
                ctx.lineWidth = 3;
                ctx.moveTo(fromx, fromy);
                ctx.lineTo(tox, toy);
                ctx.stroke();
                
                ctx.beginPath();
                ctx.fillStyle = color;
                ctx.moveTo(tox, toy);
                ctx.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                ctx.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                ctx.fill();
            }

            function drawSim() {
                ctxSim.clearRect(0, 0, canvasSim.width, canvasSim.height);
                
                const cx = canvasSim.width / 2;
                const cy = canvasSim.height / 2;
                const visual_R = 120; // Radio fijo en píxeles para visualización constante
                
                // Ángulo actual
                const theta = omega * t;
                
                // Dibuja ejes cartesianos
                ctxSim.beginPath();
                ctxSim.strokeStyle = "#e0e0e0";
                ctxSim.moveTo(0, cy); ctxSim.lineTo(canvasSim.width, cy);
                ctxSim.moveTo(cx, 0); ctxSim.lineTo(cx, canvasSim.height);
                ctxSim.stroke();

                // Trayectoria circular
                ctxSim.beginPath();
                ctxSim.strokeStyle = "#aaa";
                ctxSim.setLineDash([5, 5]);
                ctxSim.arc(cx, cy, visual_R, 0, 2 * Math.PI);
                ctxSim.stroke();
                ctxSim.setLineDash([]);

                // Posición del objeto
                const obj_x = cx + visual_R * Math.cos(theta);
                const obj_y = cy - visual_R * Math.sin(theta); // Resta porque el eje Y del canvas va hacia abajo

                // Cálculo de direcciones normalizadas para los vectores
                // Velocidad (tangente) -> v_x = -v * sin(theta), v_y = v * cos(theta)
                const dir_vx = -Math.sin(theta);
                const dir_vy = Math.cos(theta);
                
                // Aceleración (al centro) -> a_x = -cos(theta), a_y = -sin(theta)
                const dir_ax = -Math.cos(theta);
                const dir_ay = -Math.sin(theta);

                // Longitudes fijas en pantalla para vectores (para que siempre se vean bien)
                const vec_len = 65; 

                // Dibujar Vector Aceleración (Rojo, apunta al centro)
                drawArrow(
                    ctxSim, 
                    obj_x, obj_y, 
                    obj_x + dir_ax * vec_len, obj_y - dir_ay * vec_len, 
                    "#dc3545"
                );

                // Dibujar Vector Velocidad (Azul, tangente)
                drawArrow(
                    ctxSim, 
                    obj_x, obj_y, 
                    obj_x + dir_vx * vec_len, obj_y - dir_vy * vec_len, 
                    "#0d6efd"
                );

                // Dibujar Objeto (Bola)
                ctxSim.beginPath();
                ctxSim.fillStyle = "#333";
                ctxSim.arc(obj_x, obj_y, 10, 0, Math.PI * 2);
                ctxSim.fill();
                
                // Leyenda en el canvas
                ctxSim.fillStyle = "#000";
                ctxSim.font = "12px Arial";
                ctxSim.fillText("Vector Velocidad (v)", 10, 20);
                ctxSim.fillStyle = "#0d6efd";
                ctxSim.fillRect(130, 10, 20, 10);
                
                ctxSim.fillStyle = "#000";
                ctxSim.fillText("Vector Aceleración (ac)", 10, 40);
                ctxSim.fillStyle = "#dc3545";
                ctxSim.fillRect(145, 30, 20, 10);
            }

            function drawGraphs() {
                ctxG.clearRect(0, 0, canvasGraph.width, canvasGraph.height);
                
                let marginX = 50;
                let marginY = 30;
                let graphW = 520; 
                let graphH = 90; // Altura para cada una de las 3 gráficas
                let gapY = 35;
                
                let offPosY = marginY;                  // Gráfica de Posición
                let offVelY = offPosY + graphH + gapY;  // Gráfica de Velocidad
                let offAccY = offVelY + graphH + gapY;  // Gráfica de Aceleración

                // --- Ejes Comunes ---
                ctxG.strokeStyle = "#000";
                ctxG.fillStyle = "#000";
                ctxG.font = "12px Arial";

                [
                    {y: offPosY, title: "Posición (m)", color: "#198754"}, 
                    {y: offVelY, title: "Velocidad (m/s)", color: "#0d6efd"}, 
                    {y: offAccY, title: "Aceleración (m/s²)", color: "#dc3545"}
                ].forEach(ax => {
                    let midY = ax.y + graphH / 2;
                    // Eje vertical y horizontal
                    ctxG.beginPath(); ctxG.moveTo(marginX, ax.y); ctxG.lineTo(marginX, ax.y + graphH); ctxG.stroke();
                    ctxG.beginPath(); ctxG.moveTo(marginX, midY); ctxG.lineTo(marginX + graphW, midY); ctxG.stroke();
                    
                    ctxG.fillStyle = ax.color;
                    ctxG.fillText(ax.title, marginX, ax.y - 10);
                    ctxG.fillStyle = "#000";
                    ctxG.fillText("0", marginX - 15, midY + 4);
                });
                
                ctxG.fillText("t (s)", marginX + graphW - 20, offAccY + graphH / 2 + 25);

                // Leyenda de Componentes
                ctxG.fillText("Componentes:   Línea Continua = Eje X   |   Línea Punteada = Eje Y", marginX + 150, 15);

                // Escalas
                let scaleT = graphW / tMax; 
                let scalePos = (graphH / 2) / (r * 1.2); 
                let scaleVel = (graphH / 2) / (v * 1.2);  
                let scaleAcc = (graphH / 2) / (ac * 1.2); 

                // Función auxiliar para dibujar componentes X (continua) e Y (punteada)
                function drawComponentCurves(offsetY, scaleVal, valFuncX, valFuncY, color) {
                    let midY = offsetY + graphH / 2;
                    ctxG.lineWidth = 2;
                    
                    // Curva X (Continua)
                    ctxG.beginPath(); 
                    ctxG.strokeStyle = color; 
                    ctxG.setLineDash([]);
                    for(let i=0; i<=t; i+=0.02) {
                        let px = marginX + i * scaleT;
                        let py = midY - valFuncX(i) * scaleVal;
                        i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                    } 
                    ctxG.stroke();

                    // Curva Y (Punteada)
                    ctxG.beginPath(); 
                    ctxG.strokeStyle = color; 
                    ctxG.setLineDash([5, 5]);
                    for(let i=0; i<=t; i+=0.02) {
                        let px = marginX + i * scaleT;
                        let py = midY - valFuncY(i) * scaleVal;
                        i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                    } 
                    ctxG.stroke();
                    ctxG.setLineDash([]);
                }

                // Posición: x = r*cos(wt), y = r*sin(wt)
                drawComponentCurves(offPosY, scalePos, 
                    (ti) => r * Math.cos(omega * ti), 
                    (ti) => r * Math.sin(omega * ti), 
                    "#198754"
                );

                // Velocidad: vx = -v*sin(wt), vy = v*cos(wt)
                drawComponentCurves(offVelY, scaleVel, 
                    (ti) => -v * Math.sin(omega * ti), 
                    (ti) => v * Math.cos(omega * ti), 
                    "#0d6efd"
                );

                // Aceleración: ax = -ac*cos(wt), ay = -ac*sin(wt)
                drawComponentCurves(offAccY, scaleAcc, 
                    (ti) => -ac * Math.cos(omega * ti), 
                    (ti) => -ac * Math.sin(omega * ti), 
                    "#dc3545"
                );
            }

            function animate(timestamp) {
                if (!isPlaying) return;
                if (!lastTime) lastTime = timestamp;
                
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                let speed_multiplier = parseFloat(speed_slider.value);
                t += deltaTime * speed_multiplier;
                
                if (t >= tMax) {
                    t = tMax;
                    isPlaying = false;
                    btn_play.innerText = "▶ Reproducir / Pausar";
                }
                
                time_display.innerText = t.toFixed(2);
                drawSim();
                drawGraphs();
                
                if(isPlaying) {
                    animationId = requestAnimationFrame(animate);
                }
            }

            speed_slider.addEventListener('input', () => {
                speed_val.innerText = parseFloat(speed_slider.value).toFixed(1) + 'x';
            });

            btn_update.addEventListener('click', () => {
                isPlaying = false;
                t = 0;
                lastTime = 0;
                time_display.innerText = "0.00";
                calculatePhysics();
                drawSim();
                drawGraphs();
                btn_play.innerText = "▶ Reproducir / Pausar";
            });

            btn_play.addEventListener('click', () => {
                isPlaying = !isPlaying;
                if (t >= tMax) {
                    t = 0; 
                }
                if (isPlaying) {
                    lastTime = 0; 
                    animationId = requestAnimationFrame(animate);
                }
            });

            btn_reset.addEventListener('click', () => {
                isPlaying = false;
                t = 0;
                lastTime = 0;
                time_display.innerText = "0.00";
                drawSim();
                drawGraphs();
            });
            
            calculatePhysics();
            drawSim();
            drawGraphs();
        }
        
        initSimulation();
    })();
</script>
"""

# Reemplazamos el identificador para evitar conflictos en Jupyter y mostramos
html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))

In [2]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Origen de la Aceleración Centrípeta</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Observa cómo la resta de dos vectores de velocidad en instantes diferentes (<strong>Δv = v₂ - v₁</strong>) da origen a la aceleración. Reduce <strong>Δt</strong> para ver cómo el vector aceleración media converge a la aceleración centrípeta instantánea.
    </p>
    
    <!-- Controles -->
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir Giro</button>
            <button id="limit_btn_UID" style="background-color: #6f42c1; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">🎯 Animar Límite (Δt → 0)</button>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc;">
                <label style="font-weight: 600; font-size: 0.85em; margin-bottom: 5px; color: #dc3545;">Intervalo de Tiempo (Δt)</label>
                <input type="range" id="dt_slider_UID" min="0.001" max="1.5" step="0.01" value="1.0" style="width: 150px;">
                <span id="dt_val_UID" style="font-weight: bold; font-family: monospace; font-size: 0.9em;">1.000 s</span>
            </div>
        </div>
    </div>
    
    <!-- Contenedor Principal (2 Canvas) -->
    <div style="display: flex; gap: 20px; flex-wrap: wrap; justify-content: center;">
        <!-- Espacio de Posición -->
        <div style="flex: 1; min-width: 350px; max-width: 450px; text-align: center;">
            <h4 style="margin-bottom: 10px; color: #333;">Espacio de Posición</h4>
            <canvas id="canvas_pos_UID" width="400" height="400" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); width: 100%;"></canvas>
        </div>
        <!-- Espacio de Velocidades (Hodógrafa) -->
        <div style="flex: 1; min-width: 350px; max-width: 450px; text-align: center;">
            <h4 style="margin-bottom: 10px; color: #333;">Espacio de Velocidades (Δv)</h4>
            <canvas id="canvas_vel_UID" width="400" height="400" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); width: 100%;"></canvas>
        </div>
    </div>
    
    <div id="results_UID" style="margin-top: 15px; font-size: 0.9em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; justify-content: space-around;">
        <span><b style="color:#0d6efd;">▬ v₁ (Tiempo t)</b></span>
        <span><b style="color:#0dcaf0;">▬ v₂ (Tiempo t + Δt)</b></span>
        <span><b style="color:#6f42c1;">▬ Δv (Cambio vel.)</b> / a_media</span>
        <span><b style="color:#dc3545;">▬ a_c (Acel. instantánea)</b></span>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvasPos = document.getElementById('canvas_pos_UID');
            const canvasVel = document.getElementById('canvas_vel_UID');
            if (!canvasPos || !canvasVel) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctxP = canvasPos.getContext('2d');
            const ctxV = canvasVel.getContext('2d');
            
            const btn_play = document.getElementById('play_UID');
            const btn_limit = document.getElementById('limit_btn_UID');
            const dt_slider = document.getElementById('dt_slider_UID');
            const dt_val = document.getElementById('dt_val_UID');
            
            let isPlaying = false;
            let limitAnimationActive = false;
            let animationId;
            let lastTime = 0; 
            
            // Físicas Base Fijas (Para enfocar visualmente en los vectores)
            const r = 2.0;       // Radio
            const omega = 1.0;   // Velocidad angular (1 rad/s)
            const v = omega * r;
            const ac = omega * omega * r;
            
            let t = 0;           // Tiempo actual
            let dt = parseFloat(dt_slider.value); // Intervalo delta t
            
            // Dibuja flechas con estilo en el canvas
            function drawArrow(ctx, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const headlen = 10;
                const angle = Math.atan2(toy - fromy, tox - fromx);
                
                ctx.beginPath();
                ctx.strokeStyle = color;
                ctx.lineWidth = lineWidth;
                if(dashed) ctx.setLineDash([5, 5]);
                else ctx.setLineDash([]);
                
                ctx.moveTo(fromx, fromy);
                ctx.lineTo(tox, toy);
                ctx.stroke();
                ctx.setLineDash([]); // Reset
                
                ctx.beginPath();
                ctx.fillStyle = color;
                ctx.moveTo(tox, toy);
                ctx.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                ctx.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                ctx.fill();
            }

            function drawSim() {
                // --- CÁLCULOS FÍSICOS (Coordenadas Reales) ---
                const t2 = t + dt;
                const theta1 = omega * t;
                const theta2 = omega * t2;
                
                // Posiciones
                const x1 = r * Math.cos(theta1),  y1 = r * Math.sin(theta1);
                const x2 = r * Math.cos(theta2),  y2 = r * Math.sin(theta2);
                
                // Velocidades (v = w x r -> vx = -w*r*sin, vy = w*r*cos)
                const v1x = -v * Math.sin(theta1), v1y = v * Math.cos(theta1);
                const v2x = -v * Math.sin(theta2), v2y = v * Math.cos(theta2);
                
                // Variación de velocidad y aceleración media
                const dvx = v2x - v1x, dvy = v2y - v1y;
                const amx = dvx / dt,  amy = dvy / dt;
                
                // Aceleración instantánea exacta en t1
                const acx = -ac * Math.cos(theta1), acy = -ac * Math.sin(theta1);
                
                // --- RENDERIZADO ESPACIO DE POSICIÓN ---
                ctxP.clearRect(0, 0, canvasPos.width, canvasPos.height);
                const cxP = canvasPos.width / 2;
                const cyP = canvasPos.height / 2;
                
                // Factores de escala para visualización en Posición
                const Sp = 120 / r;           // Escala posición (px/m)
                const Sv = 70 / v;            // Escala velocidad (px/(m/s))
                const Sa = 80 / ac;           // Escala aceleración (px/(m/s²))

                // Ejes
                ctxP.strokeStyle = "#eee"; ctxP.lineWidth = 1;
                ctxP.beginPath(); ctxP.moveTo(0, cyP); ctxP.lineTo(canvasPos.width, cyP);
                ctxP.moveTo(cxP, 0); ctxP.lineTo(cxP, canvasPos.height); ctxP.stroke();

                // Trayectoria
                ctxP.beginPath(); ctxP.strokeStyle = "#aaa"; ctxP.setLineDash([4, 4]);
                ctxP.arc(cxP, cyP, r * Sp, 0, 2 * Math.PI); ctxP.stroke(); ctxP.setLineDash([]);

                // Coordenadas en canvas (Recordar que el eje Y de canvas es invertido)
                const c_x1 = cxP + x1 * Sp, c_y1 = cyP - y1 * Sp;
                const c_x2 = cxP + x2 * Sp, c_y2 = cyP - y2 * Sp;

                // Dibujar Vectores Velocidad (desde sus posiciones)
                drawArrow(ctxP, c_x1, c_y1, c_x1 + v1x * Sv, c_y1 - v1y * Sv, "#0d6efd"); // v1 (Azul)
                if (dt > 0.01) { // Mostrar v2 solo si no están encima
                    drawArrow(ctxP, c_x2, c_y2, c_x2 + v2x * Sv, c_y2 - v2y * Sv, "#0dcaf0", 2, true); // v2 (Cyan)
                    
                    // Bolita 2 (Sombra)
                    ctxP.beginPath(); ctxP.fillStyle = "rgba(0,0,0,0.3)";
                    ctxP.arc(c_x2, c_y2, 7, 0, Math.PI * 2); ctxP.fill();
                }

                // Dibujar Vector Aceleración Exacta (desde posición 1 hacia el centro)
                drawArrow(ctxP, c_x1, c_y1, c_x1 + acx * Sa, c_y1 - acy * Sa, "#dc3545");

                // Bolita 1 (Principal)
                ctxP.beginPath(); ctxP.fillStyle = "#333";
                ctxP.arc(c_x1, c_y1, 10, 0, Math.PI * 2); ctxP.fill();

                // --- RENDERIZADO ESPACIO DE VELOCIDADES ---
                ctxV.clearRect(0, 0, canvasVel.width, canvasVel.height);
                const cxV = canvasVel.width / 2;
                const cyV = canvasVel.height / 2;
                
                // Escalas para Espacio de Velocidades
                const Sv2 = 130 / v;           // Velocidades parten del origen
                const Sa2 = 130 / ac;          // Para comparar aceleración media y exacta
                
                // Círculo de velocidades constantes (|v| es constante)
                ctxV.beginPath(); ctxV.strokeStyle = "#ddd"; ctxV.setLineDash([2, 4]);
                ctxV.arc(cxV, cyV, v * Sv2, 0, 2 * Math.PI); ctxV.stroke(); ctxV.setLineDash([]);
                
                // Vectores v1 y v2 desde el origen
                const cv_x1 = cxV + v1x * Sv2, cv_y1 = cyV - v1y * Sv2;
                const cv_x2 = cxV + v2x * Sv2, cv_y2 = cyV - v2y * Sv2;
                
                drawArrow(ctxV, cxV, cyV, cv_x1, cv_y1, "#0d6efd"); // v1
                
                if (dt > 0.01) {
                    drawArrow(ctxV, cxV, cyV, cv_x2, cv_y2, "#0dcaf0"); // v2
                    // Vector Δv (cierra el triángulo de v1 a v2)
                    drawArrow(ctxV, cv_x1, cv_y1, cv_x2, cv_y2, "#6f42c1", 2);
                    
                    // Texto cerca de Δv
                    ctxV.fillStyle = "#6f42c1"; ctxV.font = "bold 14px Arial";
                    ctxV.fillText("Δv", cv_x1 + (cv_x2 - cv_x1)/2 + 10, cv_y1 + (cv_y2 - cv_y1)/2);
                }

                // Vector Aceleración Instantánea (Exacta)
                drawArrow(ctxV, cxV, cyV, cxV + acx * Sa2, cyV - acy * Sa2, "rgba(220, 53, 69, 0.4)", 6);

                // Vector Aceleración Media (Δv / Δt) desde el origen
                // A medida que dt -> 0, este vector se alinea perfectamente con la aceleración exacta
                drawArrow(ctxV, cxV, cyV, cxV + amx * Sa2, cyV - amy * Sa2, "#6f42c1", 3);
                
                // Etiquetas en Canvas Derecho
                ctxV.fillStyle = "#333"; ctxV.font = "12px Arial";
                ctxV.fillText("Origen", cxV - 20, cyV + 20);
                
                ctxV.fillStyle = "rgba(220, 53, 69, 0.8)";
                ctxV.fillText("a_instantánea (Fondo rojo)", 10, 20);
                
                ctxV.fillStyle = "#6f42c1";
                ctxV.fillText("a_media = Δv/Δt (Morado)", 10, 40);
            }

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                if (isPlaying) {
                    t += deltaTime * 0.5; // Velocidad de giro
                }
                
                if (limitAnimationActive) {
                    dt = dt * 0.96; // Shrink dt exponencialmente
                    if (dt <= 0.005) {
                        dt = 0.001; // Límite infinitesimal
                        limitAnimationActive = false;
                        btn_limit.innerText = "🎯 Animar Límite (Δt → 0)";
                        btn_limit.style.backgroundColor = "#6f42c1";
                    }
                    dt_slider.value = dt;
                    dt_val.innerText = dt.toFixed(3) + " s";
                }
                
                drawSim();
                
                if(isPlaying || limitAnimationActive) {
                    animationId = requestAnimationFrame(animate);
                }
            }

            dt_slider.addEventListener('input', () => {
                dt = parseFloat(dt_slider.value);
                dt_val.innerText = dt.toFixed(3) + " s";
                limitAnimationActive = false;
                btn_limit.innerText = "🎯 Animar Límite (Δt → 0)";
                btn_limit.style.backgroundColor = "#6f42c1";
                if(!isPlaying && !limitAnimationActive) drawSim();
            });

            btn_play.addEventListener('click', () => {
                isPlaying = !isPlaying;
                if (isPlaying) {
                    btn_play.innerText = "⏸ Pausar Giro";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                    lastTime = 0; 
                    animationId = requestAnimationFrame(animate);
                } else {
                    btn_play.innerText = "▶ Reproducir Giro";
                    btn_play.style.backgroundColor = "#198754";
                    btn_play.style.color = "#fff";
                }
            });

            btn_limit.addEventListener('click', () => {
                if(dt <= 0.01) {
                    // Resetear slider para repetir
                    dt = 1.0;
                    dt_slider.value = dt;
                }
                limitAnimationActive = true;
                btn_limit.innerText = "⏳ Calculando Límite...";
                btn_limit.style.backgroundColor = "#6c757d";
                lastTime = 0;
                if (!isPlaying) animationId = requestAnimationFrame(animate);
            });
            
            drawSim();
        }
        
        initSimulation();
    })();
</script>
"""

# Reemplazamos el identificador para evitar conflictos en Jupyter y mostramos
html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))